# Week 11 · Earned value: what the status report says, and what it hides

IMSE 456/656 · Program and Project Management · Larson & Gray, Chapter 13

This notebook takes a project's time-phased baseline and its weekly status reports (percent complete and actual cost for each activity), and computes every earned value measure in the chapter: PV, EV, AC, the cost and schedule variances, CPI, SPI, PCIB, ETC, EAC, VAC and TCPI. It draws the cumulative cost graph and the index trend, then checks the schedule on the network, where the dollars can mislead, and prices a recovery.

**How to use it.** Choose File → Save a copy in Drive, then Runtime → Run all. Edit only the cell marked **EDIT**. By default the notebook downloads the same conference workbook you import into Smartsheet.

| Workbook column | Meaning |
|:--|:--|
| `Activity`, `ID`, `Predecessors` | The activity, its label, and the row numbers of its predecessors |
| `Planned start`, `Planned days` | The baseline: when the activity was planned to start, and for how many workdays |
| `Budget` | The activity's baseline cost; the budgets sum to BAC |

Assumptions, as in the chapter's example: each activity is one work package; its budget is spread evenly over its planned workdays, which gives the time-phased baseline (PV); earned value uses the percent complete rule; workdays run Monday to Friday with no holidays.

In [ ]:
# ---------------------------------------------------------------- EDIT
DATA_SOURCE = "course"     # "course", "upload", "table", or a file path
COURSE_FILE = "https://raw.githubusercontent.com/harunpirim/IMSE456/main/files/week-11-evm-workbook.xlsx"
START_DATE = "2026-11-02"  # workday 1 of the project
INDIRECT_PER_DAY = 1000    # indirect cost per workday of project duration (Week 10); not in the EVM baseline

# Weekly status reports, cumulative to the end of each week (week w ends on workday 5w).
#   ID: (percent complete as a fraction, actual cost to date, duration now expected in workdays)
# The third number is the activity's actual duration if finished, or its expected total if not.
# Leave it out to keep the planned duration. Activities not listed have not started.
STATUS = {
    1: {"A": (1.00, 2100), "B": (0.70, 20000), "C": (0.30, 2000)},
    2: {"A": (1.00, 2100), "B": (1.00, 28500, 4), "C": (0.50, 3800, 11), "D": (1.00, 7600)},
    3: {"A": (1.00, 2100), "B": (1.00, 28500, 4), "C": (1.00, 6400, 11), "D": (1.00, 7600),
        "E": (0.25, 3200), "F": (0.40, 900)},
    # 4: {...},   # Studio 11, Part 2: add the week-4 report here
}
STATUS_WEEK = 3            # which report sections 2, 3, 5, 6 and 7 analyse

# Crash data from Week 10, for section 7: ID: (crash days, cost per day saved)
CRASH = {"A": (1, 1200), "B": (3, 700), "C": (4, 400), "D": (3, 900), "E": (5, 600),
         "F": (3, 300), "G": (2, 1500), "H": (2, 800), "I": (2, 0)}

# Used when DATA_SOURCE = "table", and as the fallback if the download fails
#   id,  activity,                     predecessors, planned days, budget
TABLE = [
    ("A", "Approve event brief",        [],          2,  2000),
    ("B", "Contract venue",             ["A"],       5, 30000),
    ("C", "Draft program",              ["A"],       6,  6000),
    ("D", "Configure registration",     ["B"],       4,  8000),
    ("E", "Confirm speakers",           ["C"],       8, 12000),
    ("F", "Recruit exhibitors",         ["C"],       5,  2500),
    ("G", "Publish program",            ["D", "E"],  3,  3000),
    ("H", "Prepare on-site materials",  ["F", "G"],  4,  4000),
    ("I", "Complete rehearsal",         ["H"],       2,  2500),
]

## 0 · Load the baseline

Check that the table below matches the workbook before going on. The planned start of each activity is its early start in the baseline network.

In [ ]:
import io, urllib.request, itertools
import numpy as np, pandas as pd

def from_excel(src):
    df = pd.read_excel(src, sheet_name=0); df.columns = [str(c).strip() for c in df.columns]
    need = ["Activity", "ID", "Predecessors", "Planned days", "Budget"]
    missing = [c for c in need if c not in df.columns]
    assert not missing, f"the workbook is missing column(s) {missing}; it has {list(df.columns)}"
    df = df[df["ID"].notna()].reset_index(drop=True); ids = [str(x).strip() for x in df["ID"]]; rows = []
    for k, r in df.iterrows():
        pred = []
        if pd.notna(r["Predecessors"]):
            for tok in str(r["Predecessors"]).replace(";", ",").split(","):
                tok = tok.strip()
                if not tok: continue
                if tok in ids: pred.append(tok)
                else:
                    n = int(float(tok)); assert 1 <= n <= len(ids), f"row {k + 1}: predecessor row {n} does not exist"
                    pred.append(ids[n - 1])
        rows.append((ids[k], str(r["Activity"]), pred, int(r["Planned days"]), float(r["Budget"])))
    return rows

if DATA_SOURCE == "course":
    try:
        TABLE = from_excel(io.BytesIO(urllib.request.urlopen(COURSE_FILE, timeout=30).read())); print("Loaded the conference workbook from the course repository.")
    except Exception as e:
        print(f"Could not download the course workbook ({e}); using TABLE from the EDIT cell.")
elif DATA_SOURCE == "upload":
    from google.colab import files
    up = files.upload(); name = next(iter(up)); TABLE = from_excel(io.BytesIO(up[name])); print(f"Loaded {name}.")
elif DATA_SOURCE != "table":
    TABLE = from_excel(DATA_SOURCE); print(f"Loaded {DATA_SOURCE}.")
else:
    print("Using TABLE from the EDIT cell.")

ACT = {a: dict(name=nm, pred=p, n=n, budget=b) for a, nm, p, n, b in TABLE}
IDS = list(ACT)
BAC = sum(x["budget"] for x in ACT.values())

def cpm(dur):
    es, ef = {}, {}
    for a in IDS:  # rows are in precedence order, as in the workbook
        es[a] = max([ef[p] for p in ACT[a]["pred"]], default=0); ef[a] = es[a] + dur[a]
    return es, ef

PLANNED = {a: ACT[a]["n"] for a in IDS}
ES, EF = cpm(PLANNED)
T_PLAN = max(EF.values())
def date_of(day):  # workday number (1 = START_DATE) -> date
    return pd.Timestamp(np.busday_offset(np.datetime64(START_DATE), int(day) - 1, roll="forward"))
pd.DataFrame({"activity": [ACT[a]["name"] for a in IDS], "predecessors": [", ".join(ACT[a]["pred"]) or "—" for a in IDS],
              "planned start": [f"{date_of(ES[a] + 1):%a %d %b}" for a in IDS], "planned days": [ACT[a]["n"] for a in IDS],
              "budget": [ACT[a]["budget"] for a in IDS]}, index=IDS).style.format({"budget": "${:,.0f}"})

In [ ]:
print(f"BAC (budget at completion) = ${BAC:,.0f}")
print(f"Planned duration: {T_PLAN} workdays, finishing {date_of(T_PLAN):%a %d %b %Y}")

## 1 · The baseline: planned value, time-phased

Each activity's budget is spread evenly over its planned workdays, starting at its planned start. The running total is **PV**, the baseline every status report is measured against. The chapter calls it the anchor.

In [ ]:
def pv_task(a, day):
    return ACT[a]["budget"] * min(1.0, max(0.0, (day - ES[a]) / ACT[a]["n"]))
def pv_at(day):
    return sum(pv_task(a, day) for a in IDS)

weeks = range(1, -(-T_PLAN // 5) + 1)
pd.DataFrame({"week ends": [f"{date_of(min(5 * w, T_PLAN)):%a %d %b}" for w in weeks],
              "PV this week": [pv_at(5 * w) - pv_at(5 * (w - 1)) for w in weeks],
              "cumulative PV": [pv_at(5 * w) for w in weeks]}, index=[f"week {w}" for w in weeks]).style.format("${:,.0f}", subset=["PV this week", "cumulative PV"])

## 2 · The status report

For the week chosen in `STATUS_WEEK`: each activity's percent complete times its budget is its **earned value**, the budgeted cost of the work actually done. Compare it with what was planned by now (PV) and what it actually cost (AC).

- **CV = EV − AC**: negative means the work done cost more than its budget.
- **SV = EV − PV**: negative means less work is done than planned, measured in dollars.

In [ ]:
def report(w):
    s = STATUS[w]; day = 5 * w; rows = []
    for a in IDS:
        pct, ac = (s[a][0], s[a][1]) if a in s else (0.0, 0.0)
        ev, pv = pct * ACT[a]["budget"], pv_task(a, day)
        rows.append(dict(ID=a, activity=ACT[a]["name"], pct=pct, EV=ev, AC=ac, PV=pv, CV=ev - ac, SV=ev - pv))
    df = pd.DataFrame(rows).set_index("ID")
    return df

def totals(w):
    df = report(w); EV, AC, PV = df["EV"].sum(), df["AC"].sum(), df["PV"].sum()
    CPI, SPI = EV / AC, EV / PV
    ETC = (BAC - EV) / CPI; EAC = AC + ETC
    return dict(week=w, day=5 * w, PV=PV, EV=EV, AC=AC, CV=EV - AC, SV=EV - PV, CPI=CPI, SPI=SPI, PCIB=EV / BAC,
                ETC=ETC, EAC=EAC, VAC=BAC - EAC, TCPI=(BAC - EV) / (BAC - AC))

assert STATUS_WEEK in STATUS, f"STATUS has no week {STATUS_WEEK}; it has {sorted(STATUS)}"
for w, s in STATUS.items():
    for a in s: assert a in ACT, f"week {w}: {a} is not an activity"
R = report(STATUS_WEEK); T = totals(STATUS_WEEK)
print(f"Status report, end of week {STATUS_WEEK} ({date_of(T['day']):%a %d %b}, workday {T['day']})")
show = R.copy(); show.loc["Total"] = ["", np.nan, T["EV"], T["AC"], T["PV"], T["CV"], T["SV"]]
show.style.format({"pct": "{:.0%}", "EV": "${:,.0f}", "AC": "${:,.0f}", "PV": "${:,.0f}", "CV": "{:+,.0f}", "SV": "{:+,.0f}"}, na_rep="")

## 3 · Indexes and forecasts

| Measure | Formula | Reads as |
|:--|:--|:--|
| CPI | EV ÷ AC | value earned per dollar spent |
| SPI | EV ÷ PV | value earned per dollar of work scheduled |
| PCIB | EV ÷ BAC | share of the budgeted work done |
| ETC | (BAC − EV) ÷ CPI | cost to finish, if the CPI holds |
| EAC | AC + ETC | forecast cost at completion |
| VAC | BAC − EAC | forecast over- (−) or underrun (+) |
| TCPI | (BAC − EV) ÷ (BAC − AC) | efficiency needed on the remaining work to finish on budget |

In [ ]:
for k, fmt in (("PV", "${:,.0f}"), ("EV", "${:,.0f}"), ("AC", "${:,.0f}"), ("CV", "{:+,.0f}"), ("SV", "{:+,.0f}"),
               ("CPI", "{:.3f}"), ("SPI", "{:.3f}"), ("PCIB", "{:.1%}"), ("ETC", "${:,.0f}"), ("EAC", "${:,.0f}"),
               ("VAC", "{:+,.0f}"), ("TCPI", "{:.3f}")):
    print(f"{k:>5}  {fmt.format(T[k])}")
print()
print(f"Each dollar spent has earned ${T['CPI']:.2f} of planned work; each dollar of work scheduled has been earned at ${T['SPI']:.2f}.")
print(f"If the CPI holds, the project finishes at ${T['EAC']:,.0f}, {'under' if T['VAC'] >= 0 else 'over'} budget by ${abs(T['VAC']):,.0f}.")

## 4 · The graphs

Left, the chapter's cost/schedule graph (its Figures 13.4 and 13.9): the baseline PV to the planned finish, with EV and AC at every report. Right, the index trend (its Figure 13.12).

In [ ]:
import matplotlib.pyplot as plt
GREEN, GRAY, RED, INK = "#0F5A4B", "#627571", "#B5543A", "#203331"
TT = pd.DataFrame([totals(w) for w in sorted(STATUS)]).set_index("week")
fig, (ax, bx) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
days = np.arange(0, T_PLAN + 1)
ax.plot(days, [pv_at(d) for d in days], ls="--", color=GRAY, lw=2.2, label="PV, baseline")
ax.plot([0] + list(TT["day"]), [0] + list(TT["EV"]), "-o", color=GREEN, lw=2.6, ms=5, label="EV, earned")
ax.plot([0] + list(TT["day"]), [0] + list(TT["AC"]), ":o", color=RED, lw=2.4, ms=5, label="AC, actual")
ax.axhline(BAC, color=GRAY, lw=0.8); ax.text(1, BAC * 1.01, f"BAC ${BAC:,.0f}", color=GRAY, fontsize=9, va="bottom")
ax.axvline(T["day"], color=INK, lw=0.8, ls=":"); ax.text(T["day"], ax.get_ylim()[1] * 0.02, f" week {STATUS_WEEK}", fontsize=9)
ax.set_xlabel(f"Workdays from {date_of(1):%a %d %b}"); ax.set_ylabel("Cumulative cost ($)"); ax.set_ylim(0, BAC * 1.12); ax.legend(frameon=False, loc="lower right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:,.0f}k")); ax.spines[["top", "right"]].set_visible(False)
bx.plot(TT.index, TT["CPI"], "-o", color=GREEN, lw=2.4, label="CPI = EV ÷ AC")
bx.plot(TT.index, TT["SPI"], "--o", color=GRAY, lw=2.2, label="SPI = EV ÷ PV")
bx.plot(TT.index, TT["PCIB"], ":o", color=RED, lw=2.2, label="PCIB = EV ÷ BAC")
bx.axhline(1, color=INK, lw=0.8, ls="--"); bx.set_xticks(list(TT.index)); bx.set_xlabel("Status report, week"); bx.set_ylabel("Index")
bx.set_ylim(0, max(1.2, TT[["CPI", "SPI"]].max().max() + 0.05)); bx.legend(frameon=False, loc="lower right"); bx.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
TT[["PV", "EV", "AC", "CV", "SV", "CPI", "SPI", "PCIB", "EAC", "VAC", "TCPI"]].style.format(
    {"PV": "${:,.0f}", "EV": "${:,.0f}", "AC": "${:,.0f}", "CV": "{:+,.0f}", "SV": "{:+,.0f}", "CPI": "{:.3f}", "SPI": "{:.3f}",
     "PCIB": "{:.1%}", "EAC": "${:,.0f}", "VAC": "{:+,.0f}", "TCPI": "{:.3f}"})

## 5 · What the dollars hide: the schedule on the network

SV is in dollars, not days. A large on-time activity can earn enough to hide a small late one on the critical path. Only the network can say when the project will finish. Here the network is recomputed with each activity's actual or expected duration from the status report.

The EVM baseline also holds only direct cost, as the chapter's baselines do. The indirect cost of every extra day (Week 10) is not in EAC, so this section adds it.

In [ ]:
def current_durations(w):
    d = dict(PLANNED)
    for a, v in STATUS[w].items():
        if len(v) > 2: d[a] = int(v[2])
    return d
DUR = current_durations(STATUS_WEEK); ES2, EF2 = cpm(DUR); T_FC = max(EF2.values())
slip = T_FC - T_PLAN
print(f"Planned finish:  workday {T_PLAN}, {date_of(T_PLAN):%a %d %b}")
print(f"Forecast finish: workday {T_FC}, {date_of(T_FC):%a %d %b}  ({slip:+d} workdays, {slip / T_PLAN:+.0%} of the plan)")
print(f"SPI says {1 - T['SPI']:.0%} of the scheduled work is missing; SV = {T['SV']:+,.0f} dollars.")
print()
plan_total = BAC + INDIRECT_PER_DAY * T_PLAN; fc_total = T["EAC"] + INDIRECT_PER_DAY * T_FC
print(f"Planned total cost: BAC ${BAC:,.0f} + indirect ${INDIRECT_PER_DAY * T_PLAN:,.0f} = ${plan_total:,.0f}")
print(f"Forecast total:     EAC ${T['EAC']:,.0f} + indirect ${INDIRECT_PER_DAY * T_FC:,.0f} = ${fc_total:,.0f}  ({fc_total - plan_total:+,.0f})")
pd.DataFrame({"planned days": [PLANNED[a] for a in IDS], "now expected": [DUR[a] for a in IDS],
              "planned finish": [f"{date_of(EF[a]):%a %d %b}" for a in IDS], "forecast finish": [f"{date_of(EF2[a]):%a %d %b}" for a in IDS],
              "finish moved (workdays)": [EF2[a] - EF[a] for a in IDS]}, index=IDS)

## 6 · Earning rules: how much credit for work in progress?

The percent complete rule needs an estimate for every activity in progress. The chapter's simpler rules need only *started* and *finished*: **0/100** earns nothing until the activity is finished; **50/50** earns half when it starts and half when it finishes. The same status report gives different EV, and so different CPI and SPI, under each rule.

In [ ]:
def ev_rule(w, rule):
    out = {}
    for a in IDS:
        pct = STATUS[w][a][0] if a in STATUS[w] else 0.0
        b = ACT[a]["budget"]
        out[a] = b * pct if rule == "percent complete" else b * (1.0 if pct >= 1 else (0.5 if pct > 0 and rule == "50/50" else 0.0))
    return out
rows = {}
for rule in ("percent complete", "50/50", "0/100"):
    ev = sum(ev_rule(STATUS_WEEK, rule).values())
    rows[rule] = dict(EV=ev, CPI=ev / T["AC"], SPI=ev / T["PV"])
pd.DataFrame(rows).T.style.format({"EV": "${:,.0f}", "CPI": "{:.3f}", "SPI": "{:.3f}"})

## 7 · Taking action: price a recovery

Step 4 of the control process. Using Week 10's crash data, this section tries every way of shortening the activities that are not yet finished (never below crash time, and leaving at least one day of any activity in progress) and reports the cheapest way to reach each finish. A day recovered is worth `INDIRECT_PER_DAY`.

In [ ]:
def max_cut(a, w):
    d = DUR[a]; crash_days, rate = CRASH.get(a, (d, 0))
    if rate <= 0: return 0
    pct = STATUS[w][a][0] if a in STATUS[w] else 0.0
    if pct >= 1: return 0
    remaining = int(round(d * (1 - pct)))
    return max(0, min(d - crash_days, remaining - 1 if pct > 0 else d - crash_days))
LIM = {a: max_cut(a, STATUS_WEEK) for a in IDS}
open_ids = [a for a in IDS if LIM[a] > 0]
best = {}
for cuts in itertools.product(*[range(LIM[a] + 1) for a in open_ids]):
    d = dict(DUR); cost = 0
    for a, k in zip(open_ids, cuts):
        d[a] -= k; cost += k * CRASH[a][1]
    Tn = max(cpm(d)[1].values())
    if Tn not in best or cost < best[Tn][0]:
        best[Tn] = (cost, " + ".join(f"{a}×{k}" for a, k in zip(open_ids, cuts) if k) or "—")
rec = pd.DataFrame([dict(workdays=t, finish=f"{date_of(t):%a %d %b}", crash=v[1], crash_cost=v[0],
                         indirect=INDIRECT_PER_DAY * t, crash_plus_indirect=v[0] + INDIRECT_PER_DAY * t)
                    for t, v in sorted(best.items(), reverse=True)])
opt = rec.loc[rec["crash_plus_indirect"].idxmin()]
print(f"Can still shorten: " + ", ".join(f"{a} by up to {LIM[a]} at ${CRASH[a][1]:,}/day" for a in open_ids))
print(f"Cheapest total: {opt['workdays']} workdays ({opt['finish']}) by crashing {opt['crash']} for ${opt['crash_cost']:,.0f}")
rec.style.format({"crash_cost": "${:,.0f}", "indirect": "${:,.0f}", "crash_plus_indirect": "${:,.0f}"}).hide(axis="index")

## Try it

- **Add a week.** Add the week-4 report to `STATUS` in the EDIT cell, set `STATUS_WEEK = 4` and run all. Watch whether the CPI and EAC move, and whether SPI and the network forecast agree.
- **The chapter's example.** The Digital Camera project is not a weekly workbook, but its period-7 totals check the formulas in section 3: EV 160, AC 230, PV 200, BAC 320 give CPI .70, SPI .80, EAC 460 and TCPI 1.78.
- **Your final project.** Fill in a copy of the workbook with your own baseline, upload it with `DATA_SOURCE = "upload"`, and enter a status report.